# Module 1, Notebook 1: Policy Evaluation on GridWorld

## Learning Objectives
By completing this notebook, you will:
1. Implement iterative policy evaluation from scratch for the 4x4 GridWorld
2. Track convergence sweep-by-sweep using the max delta statistic
3. Visualize how the value function evolves from uniform zero to the true $V^\pi$
4. Observe how the discount factor $\gamma$ affects both the value function shape and convergence speed
5. Understand why the contraction mapping property guarantees convergence

## Prerequisites
- Module 0, Notebook 3: Bellman equations, linear solve, iterative evaluation concept
- Familiarity with NumPy matrix operations

## Estimated Time: 15 minutes

---

**Key Insight:** Policy evaluation answers the question: "If I follow policy $\pi$ forever, what is each state worth?" The answer comes from the Bellman operator applied repeatedly until convergence. This section makes that process transparent by showing you every intermediate value function.

## 1. Setup

In [ ]:
# Purpose: Imports, random seed, and plotting configuration
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

np.random.seed(42)

# Global plotting style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size']  = 10

print("Ready. NumPy version:", np.__version__)

## 2. The 4x4 GridWorld Environment

We use Sutton & Barto's canonical 4x4 GridWorld. Two terminal states (top-left and bottom-right corners) absorb the agent with reward 0. All other transitions yield reward -1, giving the agent incentive to reach a terminal state quickly.

**Actions**: UP (0), DOWN (1), LEFT (2), RIGHT (3). Hitting a wall keeps the agent in place (still costs -1).

This is the standard setup used in S&B Chapter 4. We implement it from scratch so every operation is visible.

In [ ]:
# Purpose: Build the canonical 4x4 GridWorld MDP
# Key Concept: Terminal states are absorbing; all non-terminal transitions cost -1

N_ROWS, N_COLS = 4, 4
N_STATES  = N_ROWS * N_COLS   # 16
N_ACTIONS = 4

# Action deltas: UP, DOWN, LEFT, RIGHT
DELTAS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
ACTION_NAMES = {0: 'UP', 1: 'DOWN', 2: 'LEFT', 3: 'RIGHT'}
ACTION_ARROWS = {0: '↑', 1: '↓', 2: '←', 3: '→'}

# Terminal states: corners 0 (top-left) and 15 (bottom-right)
TERMINAL_STATES = {0, 15}

def pos_to_state(r, c):
    return r * N_COLS + c

def state_to_pos(s):
    return s // N_COLS, s % N_COLS


def build_gridworld_mdp():
    """
    Build the S&B 4x4 GridWorld MDP.

    Returns
    -------
    P : np.ndarray (16, 4, 16)  — transition tensor
    R : np.ndarray (16, 4)      — expected reward matrix
    """
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))

    for s in range(N_STATES):
        # Terminal states are absorbing (zero reward, stays in place)
        if s in TERMINAL_STATES:
            for a in range(N_ACTIONS):
                P[s, a, s] = 1.0
                R[s, a]    = 0.0
            continue

        r, c = state_to_pos(s)
        for a in range(N_ACTIONS):
            dr, dc = DELTAS[a]
            nr, nc = r + dr, c + dc

            # Wall collision: stay in current cell
            if not (0 <= nr < N_ROWS and 0 <= nc < N_COLS):
                nr, nc = r, c

            s_next = pos_to_state(nr, nc)
            P[s, a, s_next] += 1.0
            R[s, a] = -1.0   # All transitions cost -1

    # Verify transition rows sum to 1
    assert np.allclose(P.sum(axis=2), 1.0), "Transition probabilities must sum to 1."
    return P, R


P_gw, R_gw = build_gridworld_mdp()

print(f"GridWorld: {N_STATES} states, {N_ACTIONS} actions")
print(f"Terminal states: {sorted(TERMINAL_STATES)} (corners)")
print(f"P shape: {P_gw.shape}   R shape: {R_gw.shape}")

# Print the grid layout
print("\nGrid layout (T=terminal, .=normal):")
for r in range(N_ROWS):
    row = []
    for c in range(N_COLS):
        s = pos_to_state(r, c)
        row.append(f'T({s:2d})' if s in TERMINAL_STATES else f'. ({s:2d})')
    print('  ' + '  |  '.join(row))

## 3. Iterative Policy Evaluation — Core Algorithm

The algorithm is simple:

```
Initialize V(s) = 0 for all s
Repeat:
    For each state s:
        V_new(s) = sum_a pi(a|s) * sum_s' P(s'|s,a) * [R(s,a,s') + gamma * V(s')]
    delta = max_s |V_new(s) - V(s)|
    V = V_new
Until delta < theta
```

We implement this with full history recording so we can later visualize every intermediate state.

In [ ]:
# Purpose: Implement iterative policy evaluation with full history recording
# Key Concept: Storing all intermediate V vectors enables convergence visualization

def policy_evaluation(
    P, R, policy, gamma=1.0, theta=1e-6, max_iter=500, record_every=1
):
    """
    Iterative policy evaluation with full history.

    Parameters
    ----------
    P : np.ndarray (n_states, n_actions, n_states)
    R : np.ndarray (n_states, n_actions)
    policy : np.ndarray (n_states, n_actions)
        Stochastic policy; rows must sum to 1.
    gamma : float
        Discount factor. Use gamma=1.0 for episodic tasks with terminal states.
    theta : float
        Convergence threshold.
    max_iter : int
    record_every : int
        Store a snapshot of V every `record_every` iterations.

    Returns
    -------
    V : np.ndarray (n_states,)
        Converged value function.
    delta_history : list of float
        Max delta per sweep.
    V_history : list of np.ndarray
        Snapshots of V at recorded iterations.
    snap_iterations : list of int
        Iteration numbers corresponding to V_history.
    """
    n_states, n_actions, _ = P.shape
    V = np.zeros(n_states)
    delta_history  = []
    V_history      = [V.copy()]
    snap_iterations = [0]

    for iteration in range(1, max_iter + 1):
        V_new = np.zeros(n_states)

        for s in range(n_states):
            # Vectorized Bellman update:
            # V_new[s] = sum_a pi(a|s) * [R[s,a] + gamma * sum_s' P[s,a,s'] * V[s']]
            expected_next = P[s].dot(V)              # shape (n_actions,)
            action_values = R[s] + gamma * expected_next  # Q-values for this state
            V_new[s] = policy[s].dot(action_values)  # weighted by policy

        max_delta = np.max(np.abs(V_new - V))
        delta_history.append(max_delta)
        V = V_new

        if iteration % record_every == 0:
            V_history.append(V.copy())
            snap_iterations.append(iteration)

        if max_delta < theta:
            # Always record the final converged V
            if snap_iterations[-1] != iteration:
                V_history.append(V.copy())
                snap_iterations.append(iteration)
            print(f"Converged in {iteration} sweeps (max delta = {max_delta:.2e})")
            break

    return V, delta_history, V_history, snap_iterations


# Uniform random policy: each action equally likely
uniform_policy = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS

print("Running policy evaluation with gamma=1.0 (undiscounted, episodic):")
V_uniform, deltas, V_snaps, snap_iters = policy_evaluation(
    P_gw, R_gw, uniform_policy, gamma=1.0, theta=1e-6, record_every=5
)

print("\nConverged V^pi (uniform policy), reshaped as 4x4 grid:")
print(V_uniform.reshape(N_ROWS, N_COLS).round(1))

## 4. Visualizing Value Function Evolution

We plot snapshots of the value function at different iterations. Early iterations show V near zero everywhere; over time the values propagate from the terminal states outward, like ripples in a pond.

The color scale is centered at zero so positive and negative values are clearly distinguished.

In [ ]:
# Purpose: Visualize V at multiple iterations to show propagation from terminal states
# Key Concept: Values propagate backwards from terminal states at rate gamma per step

def plot_V_snapshots(V_snaps, snap_iters, n_rows, n_cols, terminal_states, title):
    """
    Plot up to 6 value function snapshots as heatmaps.

    Parameters
    ----------
    V_snaps : list of np.ndarray
    snap_iters : list of int
    n_rows, n_cols : int
    terminal_states : set
    title : str
    """
    # Pick up to 6 evenly spaced snapshots
    n_snaps = min(6, len(V_snaps))
    indices = np.linspace(0, len(V_snaps) - 1, n_snaps, dtype=int)

    fig, axes = plt.subplots(1, n_snaps, figsize=(3.5 * n_snaps, 4))
    if n_snaps == 1:
        axes = [axes]

    # Compute global color range across all snapshots
    all_vals = np.concatenate([V_snaps[i] for i in indices])
    vmin, vmax = all_vals.min(), all_vals.max()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=max(vmax, 0.01))

    for ax, idx in zip(axes, indices):
        V_grid = V_snaps[idx].reshape(n_rows, n_cols)
        im = ax.imshow(V_grid, cmap='RdYlGn', norm=norm, interpolation='nearest')

        for r in range(n_rows):
            for c in range(n_cols):
                s = r * n_cols + c
                label = 'T' if s in terminal_states else f'{V_grid[r, c]:.1f}'
                color = 'white' if s in terminal_states else 'black'
                ax.text(c, r, label, ha='center', va='center',
                        fontsize=9, color=color, fontweight='bold')

        ax.set_title(f'Iter {snap_iters[idx]}', fontsize=11)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04, label='V(s)')
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('policy_eval_snapshots.png', dpi=120, bbox_inches='tight')
    plt.show()


plot_V_snapshots(
    V_snaps, snap_iters, N_ROWS, N_COLS, TERMINAL_STATES,
    title='Value Function Evolution — Uniform Random Policy ($\\gamma=1.0$)'
)

## 5. Convergence Curve — Max Delta per Sweep

The convergence curve plots $\max_s |V_{k+1}(s) - V_k(s)|$ at each sweep. On a log scale, this should decay approximately linearly at rate $\gamma$. When $\gamma = 1.0$ (undiscounted), the decay is driven by the structure of the MDP rather than geometric discounting.

In [ ]:
# Purpose: Plot the max delta convergence curve
# Key Concept: Tracking max delta tells us when it is safe to stop iterating

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Left: Delta curve ---
ax = axes[0]
ax.semilogy(deltas, color='#1c7ed6', linewidth=2)
ax.set_xlabel('Sweep', fontsize=12)
ax.set_ylabel('Max $|\\Delta V|$ (log scale)', fontsize=12)
ax.set_title('Convergence: Max Delta per Sweep', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')
ax.axhline(1e-6, color='red', linestyle='--', linewidth=1, label='$\\theta = 10^{-6}$')
ax.legend(fontsize=10)

# --- Right: Final converged V ---
ax = axes[1]
V_final_grid = V_uniform.reshape(N_ROWS, N_COLS)
vmin, vmax = V_final_grid.min(), 0
norm = TwoSlopeNorm(vmin=vmin, vcenter=(vmin + vmax) / 2, vmax=0.01)
im = ax.imshow(V_final_grid, cmap='RdYlGn', norm=norm, interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for r in range(N_ROWS):
    for c in range(N_COLS):
        s = pos_to_state(r, c)
        label = 'T' if s in TERMINAL_STATES else f'{V_final_grid[r, c]:.1f}'
        ax.text(c, r, label, ha='center', va='center', fontsize=10,
                color='white' if s in TERMINAL_STATES else 'black', fontweight='bold')
ax.set_title('Converged $V^{\\pi}$ (uniform random policy)', fontsize=12, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig('policy_eval_convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Total sweeps: {len(deltas)}")

## 6. Effect of Discount Factor on Convergence

Discount factor $\gamma$ has two effects:
1. **Value scale**: smaller $\gamma$ compresses values toward zero
2. **Convergence speed**: smaller $\gamma$ converges faster (stronger contraction)

We evaluate the same uniform policy under three values of $\gamma$ and compare the convergence curves.

In [ ]:
# Purpose: Compare convergence speed across gamma values
# Key Concept: The Bellman operator is a gamma-contraction; smaller gamma = faster convergence

gammas_to_compare = [0.5, 0.9, 1.0]
colors = ['#2f9e44', '#1c7ed6', '#e64980']

results = {}
for g in gammas_to_compare:
    V_g, deltas_g, _, _ = policy_evaluation(
        P_gw, R_gw, uniform_policy, gamma=g, theta=1e-6, record_every=10
    )
    results[g] = {'V': V_g, 'deltas': deltas_g}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Convergence curves ---
ax = axes[0]
for g, color in zip(gammas_to_compare, colors):
    ax.semilogy(results[g]['deltas'], color=color, linewidth=2, label=f'$\\gamma={g}$')
ax.set_xlabel('Sweep', fontsize=12)
ax.set_ylabel('Max $|\\Delta V|$ (log scale)', fontsize=12)
ax.set_title('Convergence Speed vs. Discount Factor', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

# --- Right: Final V values (state 5 = middle, away from terminals) ---
ax = axes[1]
state_indices = list(range(N_STATES))
x = np.arange(N_STATES)
width = 0.25

for i, (g, color) in enumerate(zip(gammas_to_compare, colors)):
    ax.bar(x + i * width, results[g]['V'], width=width, color=color,
           alpha=0.8, label=f'$\\gamma={g}$')

ax.set_xlabel('State index', fontsize=12)
ax.set_ylabel('V(s)', fontsize=12)
ax.set_title('Converged V per State\n(uniform random policy)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_xticks(x + width)
ax.set_xticklabels([str(s) for s in state_indices], fontsize=7)

plt.tight_layout()
plt.savefig('gamma_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("Sweeps to convergence:")
for g in gammas_to_compare:
    print(f"  gamma={g}: {len(results[g]['deltas'])} sweeps")

## 7. Exercises

### Exercise 1.1: Evaluate a Non-Uniform Policy

**Task:** Define a **right-biased policy** that takes action RIGHT with probability 0.7 and distributes the remaining 0.3 uniformly across the other three actions. Compute $V^\pi$ using `policy_evaluation` with $\gamma = 0.99$. Report the value at state 0 (top-left corner) and state 14 (second-to-last cell).

**Expected Output:** Two floats, V[0] and V[14], printed to the console.

**Hints:**
<details>
<summary>Hint 1</summary>
Build a (16, 4) numpy array where each row has 0.7 in column 3 (RIGHT) and 0.1 in the other three columns.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
```python
right_biased = np.ones((N_STATES, N_ACTIONS)) * 0.1  # 0.1 each for non-RIGHT
right_biased[:, ACTION_RIGHT] = 0.7
```
Then verify rows sum to 1: `assert np.allclose(right_biased.sum(axis=1), 1.0)`
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
ACTION_RIGHT = 3

# Step 1: Build the right-biased policy
right_biased_policy = None  # shape (N_STATES, N_ACTIONS)

# Step 2: Run policy evaluation
V_right_biased = None  # Converged value function

# Step 3: Extract specific state values
v_state_0  = None
v_state_14 = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_1():
    assert right_biased_policy is not None, "Define right_biased_policy!"
    assert right_biased_policy.shape == (N_STATES, N_ACTIONS), \
        f"Expected shape ({N_STATES}, {N_ACTIONS})"
    assert np.allclose(right_biased_policy.sum(axis=1), 1.0), \
        "Each row must sum to 1 (valid probability distribution)."
    assert right_biased_policy[0, ACTION_RIGHT] > 0.5, \
        "RIGHT action should have probability > 0.5 (biased right)."

    assert V_right_biased is not None, "Run policy_evaluation to get V_right_biased!"
    assert V_right_biased.shape == (N_STATES,), \
        f"V_right_biased must have shape ({N_STATES},)"

    # Terminal states should have V near 0
    assert abs(V_right_biased[0]) < 0.1, \
        f"V[0] (terminal) should be ~0, got {V_right_biased[0]:.3f}"
    assert abs(V_right_biased[15]) < 0.1, \
        f"V[15] (terminal) should be ~0, got {V_right_biased[15]:.3f}"

    assert v_state_0  is not None, "Extract v_state_0!"
    assert v_state_14 is not None, "Extract v_state_14!"

    print(f"Exercise 1.1 passed!")
    print(f"  V[0]  (top-left, terminal):        {v_state_0:.4f}")
    print(f"  V[14] (bottom-right -1, non-term): {v_state_14:.4f}")

test_exercise_1_1()

### Exercise 1.2: Convergence Threshold Analysis

**Task:** Run policy evaluation on the uniform policy with $\gamma = 0.9$ using three different convergence thresholds: $\theta \in \{0.1, 0.01, 1 \times 10^{-6}\}$. Record the number of sweeps required for each. Then compute the max absolute error against the most accurate solution (smallest $\theta$).

**Expected Output:** A table of (theta, n_sweeps, max_error) printed to the console.

**Hints:**
<details>
<summary>Hint 1</summary>
Call `policy_evaluation` three times with different `theta` values and record `len(delta_history)` for each.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
thresholds = [0.1, 0.01, 1e-6]
GAMMA_EX = 0.9

# Store results: list of (theta, n_sweeps, V)
threshold_results = []  # Fill this in

# Compute max error for each threshold relative to the most accurate (theta=1e-6)
errors = []  # Fill this in

# Print the table
# Expected format:
#   theta     sweeps   max_error
#   1.0e-01   ???      ???
#   1.0e-02   ???      ???
#   1.0e-06   ???      0.0000

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_1_2():
    assert len(threshold_results) == 3, \
        "threshold_results must have 3 entries (one per theta value)."

    thetas_found  = [r[0] for r in threshold_results]
    sweeps_found  = [r[1] for r in threshold_results]
    V_solutions   = [r[2] for r in threshold_results]

    # Sweeps should increase as theta decreases (tighter tolerance = more sweeps)
    assert sweeps_found[0] <= sweeps_found[1] <= sweeps_found[2], \
        (f"Sweeps should increase as theta decreases. Got: {sweeps_found}. "
         "Did you run with theta in decreasing order?")

    assert len(errors) == 3, "Compute max error for each theta (list of 3 floats)."
    assert errors[-1] < 1e-9, "Error against itself (theta=1e-6) should be 0."
    assert errors[0] >= errors[1] >= errors[2], \
        "Larger theta should produce larger error."

    print(f"Exercise 1.2 passed!")
    print(f"  {'theta':>10}  {'sweeps':>8}  {'max_error':>12}")
    for theta, n, err in zip(thresholds, sweeps_found, errors):
        print(f"  {theta:>10.1e}  {n:>8d}  {err:>12.6f}")

test_exercise_1_2()

## Key Takeaways

1. **Policy evaluation is the foundation of all DP methods**: you must know how good your current policy is before you can improve it.
2. **The Bellman operator is a contraction mapping** with constant $\gamma$. Starting from any $V$, repeated application converges to the unique fixed point $V^\pi$.
3. **Convergence is tracked via max delta**: $\max_s |V_{k+1}(s) - V_k(s)| < \theta$ is the standard stopping criterion. Smaller $\theta$ gives higher accuracy but requires more sweeps.
4. **Values propagate outward from terminal states**: after $k$ sweeps, states up to $k$ transitions from a terminal state have accurate values.
5. **Smaller $\gamma$ means faster convergence** but compresses the value scale. For episodic tasks with natural termination, $\gamma = 1.0$ is appropriate.
6. **The convergence threshold $\theta$ is a hyperparameter**: tighter thresholds give more accurate values at the cost of more computation.

## What's Next

Notebook 2 (`02_policy_and_value_iteration.ipynb`) combines policy evaluation with **policy improvement** to find the *optimal* policy $\pi^*$. We implement both policy iteration (evaluate fully, then improve) and value iteration (evaluate one step, then improve) and compare their convergence properties.